# 14 — Cross-cohort stability: what is stable and what depends on context

The cross-cohort analyses of the manuscript (Results 2.2, Figure 3), for **one state set** chosen
below: an HMM or a clusterer without a temporal model (mean-shift, threshold). Everything notebook
11 shows is here (sections 1–5), followed by the new stability analyses (sections 6–8).

**Read the HMM results with one caveat.** Projected cohorts are labelled with `hmm.predict`, the
Viterbi path, which uses the transition matrix learned on the training cohorts as a prior. Part of
the similarity of the transition structure across cohorts is therefore built in. A state set from
a clusterer without a temporal model (mean-shift, threshold) labels every window on its own: a
transition structure shared across cohorts is then a property of the data, not of the model.
Running this notebook on both is the check.

* **Equivalence, not "no difference"** (TOST, Lakens 2017): a variable is equivalent across groups
  when the 90% CI of the difference lies inside ± SESOI = `sesoi_d` × pooled SD.
* **Stable** (section 6) = equivalent between cohorts (movie) **and** between rest and movie in the
  same people.
* **Individual consistency** (section 7) = ICC(3,1) between the same people's rest and movie: do
  people keep their rank when the context changes?

In [ ]:
import sys
from pathlib import Path

# THE PACKAGE, which is not pip-installed inside the analysis image. A Jupyter
# kernel starts with sys.path[0] = the notebook's own directory, so without
# this `import fmri_decomposition` fails from notebooks/.
#
# The REPO ROOT and nothing else. `notebooks/` used to be on this list too,
# back when the table and figure modules lived beside the .ipynb files; that
# part is gone and must not come back -- a directory on sys.path can shadow a
# real package, which is how an ImportError turns into the wrong module.
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from fmri_decomposition.unsupervised_model import cohort_tables as ct

# ---- THE STATE SET. Any clusterer stage 4b wrote works: HMM (HMM1_/HMM2_), mean-shift
# (MeanShift_...), threshold (ThresholdCluster_...); the list printed below shows what exists.
ATLAS    = "harvardoxford"
WINDOW_S = 30.0
STATES   = "HMM2_umap3_8"          # e.g. "MeanShift_umap3_9", "ThresholdCluster_umap3_8"
SAVE_DIR = None                    # e.g. "figures/14" to save PNG + SVG
from fmri_decomposition.figures import cohorts as cp

cfg = ct.Settings(
    space=dict(atlas=ATLAS, window_s=WINDOW_S, states=STATES),
    entropy_estimator="miller_madow",   # None = entropy_rate_bits as stored (plug-in)
    # label -> the cohort= directories it reads, and its role in the state fit. A new cohort is one line.
    cohorts={"ds002837": dict(dirs=["ds002837"], role="train"),
             "cneuromod": dict(dirs=["cneuromod"], role="train"),
             "camcan": dict(dirs=["camcan", "camcan_rest"], role="projected"),
             "hcp7t_fix": dict(dirs=["hcp7t_fix"], role="projected")},
    rest_pattern=r"rest",          # a task= that IS this (whole name, any case) is rest; every other is movie
    # figure 1: movie vs rest
    alpha=0.10,                    # BH-FDR
    fc_task=1.0,                   # |log2FC| for the graph (1 = twofold)
    fc_dyn=0.25,                   # |log2FC| for the dynamics
    edge_top=12,                   # the mean's strongest arrows, per task
    # figures 2 and 4: equivalence
    sesoi_d=0.2,                   # TOST bounds, in pooled SDs
    top_transitions=8,             # figure 2: the strongest transitions shown
    save_dir=SAVE_DIR,
)
sets = ct.available_sets(cfg.root, cfg.cohorts, cfg.modelling_version)
print(f"{len(sets)} state sets have every cohort; e.g.")
for _, r in sets.head(8).iterrows():
    print(f"  space=dict(atlas={r.atlas!r}, window_s={r.window_s}, states={r.states!r})")
co = ct.Cohorts(cfg)
# What is read, from where, and as which kind: check this before reading any figure.
co.tasks()

## 1. Rest and movie, per cohort and together

Rows are the cohorts, then their mean (every cohort weighs once); columns are rest and movie. Each
diagram is the cohort's own dynamics, averaged over its subjects (a subject's runs or films first).
**Circle area** is time in a state, **the loop** is p(stay), **arrow width** is p(j | i), on one
width scale for the whole figure; every diagram draws the mean's `edge_top` strongest arrows of each
task. **Blue** on the movie is what differs from the **same people's** rest: paired t-test (Welch if
a cohort's rest and movie subjects do not overlap), BH-FDR over every variable, and |log2FC| ≥
`fc_task` (`fc_dyn` for the dynamics), with the change in pp. Beside each diagram are the dynamics:
⇄ switch rate, stopwatch = mean dwell, H entropy rate, ✳ dispersion, # distinct transitions.

In [ ]:
graphs = co.graphs()
marks, dyn = co.task_change()
cp.fig_diagrams(co, graphs, marks, dyn)
# marks[marks["hit"]].sort_values(["cohort", "p"])

## 2. Is every cohort equivalent to the others?

Movie only (the task every cohort shares). The transitions are the `top_transitions` strongest;
every occupancy and every dynamics variable is shown, the dynamics each on its own axis.

**2a, equivalence (TOST), in the variables' own units.** Each cohort's mean with its 90% CI (time in
% for the probabilities, seconds or bits for the dynamics). The grey band is the mean of the cohorts
± SESOI. **Filled** = equivalent to all the others pooled (the 90% CI of the difference lies inside
± SESOI), **hollow** = not shown equivalent; **✓** at the right when every cohort is.

**2b, effect size (Cohen's d).** The same comparison as a standardised difference with its 95% CI,
on bands at |d| 0.2 (small), 0.5 (medium), 0.8 (large). A large cohort is never "equivalent" by
accident, and a tiny one rarely is (its CI is wide); d says how far apart the cohorts are,
whatever their size.

In [ ]:
agree = co.agreement()
cp.fig_tost(co, agree)
# agree.groupby("variable")["equivalent"].all().value_counts()

In [ ]:
d = co.cohen()
cp.fig_cohen(co, d)
# d.pivot_table(index="variable", columns="cohort", values="d").round(2)

## 3. Does each projected cohort reproduce the training cohorts?

Movie, one row per projected cohort. Every variable's mean in the projected cohort (y) against its
mean in each training cohort (x, a dot per training cohort, in its colour): on the diagonal, the
projection reproduces the training. The title gives the correlation with each training cohort and
between the training cohorts: a projected cohort that matches them about as well as they match each
other is as close as two training cohorts are. The dynamics are log-log (they span 0.4 to ~35).

In [ ]:
means = co.cohort_means()
cp.fig_train_vs_projected(co, means)
# means.T

## 4. The transitions as matrices

One matrix per group (p(j | i), on one colour scale), with the time in each state on top and the
dynamics below, then a last panel testing every cell, occupancy and dynamics: each group vs all the
others pooled. Three switches:

* `GROUP_BY = "cohort"` compares the cohorts' movie; `"task"` compares tasks, with `TASKS = "kind"`
  (rest vs movie) or `"film"` (every movie task apart: films, episodes). `WITHIN = "camcan"` keeps a
  task comparison inside one cohort; `None` pools every cohort.
* `TEST = "tost"` (equivalence: ✓ where every group is equivalent to the others within ± `sesoi_d`
  SD, a red frame with k/N where only k are), `"cohen"` (the group furthest from the others, |d|
  coloured small / medium / large) or `None` (the matrices only).
* `SD = True` writes each cell's between-subject SD (pp), ± SD bars on the occupancy and mean ± SD
  for the dynamics.

Matrices that look alike can still fail the TOST: the bounds are ± `sesoi_d` × the between-subject
SD, and an HMM's p(j | i) varies little between subjects, so ± 0.2 SD can be a fraction of a pp.
Rest and movie are mostly the same people: the tests treat the groups as independent, which makes
equivalence harder to show, not easier.

In [ ]:
GROUP_BY = "cohort"            # "cohort" | "task"
TASKS = "kind"                 # with "task": "kind" (rest vs movie) | "film" (each movie task)
WITHIN = None                  # with "task": one cohort label, or None = every cohort pooled
TEST = "tost"                  # "tost" | "cohen" | None
SD = False
groups = co.groups(GROUP_BY, TASKS, WITHIN)
cp.fig_matrices(co, groups, TEST, SD, title=f"4  transitions by {GROUP_BY}", name=f"4_matrix_{GROUP_BY}")

In [ ]:
# the same figure, Cohen's d and with the SD
cp.fig_matrices(co, groups, "cohen", sd=True, title=f"4  transitions by {GROUP_BY}", name=f"4_matrix_{GROUP_BY}")

### 4, by task: rest vs movie

Every cohort's rest against every cohort's movie (`WITHIN` narrows it to one cohort). Change
`TASKS` to `"film"` to set the films (and episodes) against each other instead.

In [ ]:
task_groups = co.groups("task", tasks="kind", cohort=None)
cp.fig_matrices(co, task_groups, "cohen", title="4  rest vs movie", name="4_matrix_task_kind")

In [ ]:
film_groups = co.groups("task", tasks="film", cohort=None)
cp.fig_matrices(co, film_groups, "tost", title="4  the films against each other", name="4_matrix_task_film")

## 5. The latent space, per cohort

Every window of every cohort in the stage-4 latent space (`latents/`), one 3D panel per cohort on
the same axes. The embedding is fitted on the training cohorts and the others are projected onto
it, so a projected cohort filling the same space as the training ones is the same claim as figure 3,
seen window by window. At most `MAX_WINDOWS` windows per person are drawn (chosen at random), so a
cohort with long scans does not bury the others.

`ATLAS`, `WINDOW_S` and `STATES` pick any latents stage 4 wrote (the list above), independently of
`space`. `BY` picks the colouring: `"participant"` (a gradient, one colour per person: are the people spread
over the whole space or each in a corner?), `"task"` (rest, movie and each film), `"cluster"` (the
state of `space`, needs the clustering stage) or `"single"` (one person per cohort, `WHO =
{cohort: n or subject id}` with n = the n-th subject in alphabetical order, or one n for every
cohort, in colour with their path, over everyone else in grey; a cohort left out shows its first
subject, and the title gives each person's id and number). `PATHS = True` joins each person's windows in time order. The figure is plotly:
drag to rotate, scroll to zoom, hover for the subject and task; with `save_dir` it is also written
as an .html. Needs `pip install --user plotly`.

In [ ]:
ct.latent_options(co).atlas         # every atlas x window with latents, and their embeddings and state sets

In [ ]:
ct.latent_options(co).states[0]         # every atlas x window with latents, and their embeddings and state sets

In [ ]:
ATLAS = 'harvardoxford'                   # any atlas listed above; None = space's
WINDOW_S = None                # any window listed above (-1 = 1 TR); None = space's
STATES = 'HMM2_umap3_27'                  # any state set listed above, for BY="cluster"; None = space's
EMBEDDING = "umap3"            # or "pca3"
BY = "cluster"             # "participant" | "task" | "cluster" | "single"
WHO = {}#{"camcan": 1, "ds002837": 1, "cneuromod":1}                      # with BY="single": {cohort: n or id}, n = the n-th subject alphabetically,
                               # e.g. {"camcan": 1, "ds002837": "SUB-01"}; or one n for every cohort, e.g. WHO = 3
PATHS = False
MAX_WINDOWS = 150
lat = ct.load_latents(co, EMBEDDING, MAX_WINDOWS, atlas=ATLAS, window_s=WINDOW_S, states=STATES)
cp.fig_latents(co, lat, by=BY, who=WHO, embedding=EMBEDDING, paths=PATHS).show()
# lat.groupby(["cohort", "kind"])["sub"].nunique()

## 6. Panel A — what is stable and what depends on context

Per variable, two effect sizes with 95% CI against the equivalence band (± `sesoi_d` SD):

* **orange**, between cohorts (movie): the cohort furthest from the others (|Cohen's d|);
* **blue**, movie vs rest in the same people (Cam-CAN, HCP): |d_av| (mean difference over the
  mean of the two SDs, Lakens 2013), bootstrap CI over people.

**Stable** when both are equivalent (TOST at 5%), otherwise **context-dependent**. The variables:
the **transition structure** as one number per person (Pearson r between the person's p(j | i)
and the reference p(j | i), on the rows of states occupied ≥ 1%), the dynamics, the occupancies
and the `top_transitions` strongest transitions. Count-based variables (distinct transitions)
depend on scan length: compare them only on equal-length segments.

In [ ]:
st = co.stability(n_boot=2000)
st.round(3)

In [ ]:
cp.fig_stability_forest(co, st)

## 7. Panel B — stable in the mean, stable in the individual

x = how much the mean shifts between rest and movie (|d_av|, same people); y = whether people keep
their rank (ICC(3,1) rest ↔ movie on cohort-demeaned values, bootstrap 95% CI). Quadrants:

* **stable trait** — same mean, same ranking;
* **context-modulated trait** — the mean shifts, but people keep their rank (usable as an
  individual marker, compared within one context);
* **context-driven** — shifts, ranking not kept;
* **unreliable** — no shift, no ranking.

`PLANE_VARIABLES` keeps the plot readable; the table above has every variable, and `icc_<cohort>`
gives the ICC per cohort (Cam-CAN's wide age range tends to raise it).

In [ ]:
PLANE_VARIABLES = ["transition_fit", *co.dyn, *co.occ]
cp.fig_stability_plane(co, st, PLANE_VARIABLES)

In [ ]:
st.filter(regex="^variable$|^icc").round(2)

## 8. The reference structure: similarity across cohorts and occupancy

**8a.** Pearson r between the mean p(j | i) of every cohort and condition (rows of states occupied
≥ 1%; rare states' rows are noise). High r everywhere = one transition structure shared by every
cohort and condition. Its null (the same with the state sequences circularly shifted, which keeps
the overlap of 30-s windows) needs the sequences and is not computed here yet.

**8b.** Mean time in each state, by cohort and condition: the context-dependent counterpart.

The **reference graph** (the mean over cohorts, the "what to remember" panel) is the `mean` row of
section 1's figure.

In [ ]:
S = co.similarity()
cp.fig_similarity(co, S)

In [ ]:
O = co.occupancy_table()
cp.fig_occupancy(co, O)